# uxplain demo — Regression

End-to-end walkthrough of the regression pipeline on synthetic data:

1. Generate a regression problem with `sklearn.datasets.make_regression`.
2. Fit `UncertaintyExplanationPipeline` with the default Crepes conformal predictor.
3. Inspect prediction intervals and validate empirical coverage.
4. Explain interval width with **SHAP**, **PDP**, and **LIME**.
5. Swap in **CQR** (Conformalized Quantile Regression) as an alternative conformal backend.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split

import uxplain
from uxplain import CQRConformalPredictor, UncertaintyExplanationPipeline

print(f"uxplain {uxplain.__version__}")

RNG = 42

## 1. Simulated data

A regression problem with 6 features (4 informative) and 1 200 samples. We wrap `X` in a `DataFrame` so feature names propagate to SHAP/PDP/LIME plots automatically.

In [ ]:
X_raw, y = make_regression(
    n_samples=1200, n_features=6, n_informative=4, noise=20.0, random_state=RNG
)
feature_names = [f"x{i}" for i in range(X_raw.shape[1])]
X = pd.DataFrame(X_raw, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RNG
)
print(f"train: {X_train.shape}  test: {X_test.shape}")

## 2. Quickstart

The pipeline auto-splits 20 % of the training set as calibration. Defaults: `confidence=0.9`, `conformal_method="normalized"`, `xai_method="shap"`.

In [ ]:
pipeline = UncertaintyExplanationPipeline(
    model=RandomForestRegressor(n_estimators=200, random_state=RNG),
    confidence=0.9,
    random_state=RNG,
)
pipeline.fit(X_train, y_train)

### Prediction intervals

`predict` returns `(lower, upper)`. Verify empirical coverage on the test set.

In [ ]:
lower, upper = pipeline.predict(X_test)
covered = (lower <= y_test) & (y_test <= upper)

print(f"Empirical coverage:    {covered.mean():.2%}   (target ≥ 90 %)")
print(f"Mean interval width:   {(upper - lower).mean():.2f}")
print(f"Median interval width: {np.median(upper - lower):.2f}")

preview = pd.DataFrame({"y_true": y_test[:5], "lower": lower[:5], "upper": upper[:5]})
preview["width"] = preview["upper"] - preview["lower"]
preview.round(2)

## 3. Explain interval width with SHAP

SHAP attributes each feature's contribution to the **uncertainty metric** — `"width"` by default for regression. Larger absolute SHAP values mean the feature is driving the model's uncertainty up or down for that sample.

In [ ]:
result_shap = pipeline.explain(
    X_test.iloc[:100],
    plot_kind=["beeswarm", "bar", "waterfall"],
    waterfall_index=0,
)

## 4. PDP — global feature effects on uncertainty

PDP is global, not per-sample. We refit the pipeline with `xai_method="pdp"`. `importance` ranks features by how much they shift the metric across the input range.

In [ ]:
pipeline_pdp = UncertaintyExplanationPipeline(
    model=RandomForestRegressor(n_estimators=200, random_state=RNG),
    xai_method="pdp",
    random_state=RNG,
)
pipeline_pdp.fit(X_train, y_train)
_ = pipeline_pdp.explain(X_test.iloc[:200], plot_kind=["pdp", "importance"])

## 5. LIME — local linear approximation

LIME builds a per-sample linear surrogate around a single test point. `waterfall_index` selects which sample to inspect.

In [ ]:
pipeline_lime = UncertaintyExplanationPipeline(
    model=RandomForestRegressor(n_estimators=200, random_state=RNG),
    xai_method="lime",
    n_lime_samples=3000,
    random_state=RNG,
)
pipeline_lime.fit(X_train, y_train)
_ = pipeline_lime.explain(X_test.iloc[:5], waterfall_index=0)

## 6. Alternative conformal backend — CQR

Conformalized Quantile Regression (Romano et al., 2019) uses two quantile regressors instead of residual-based scoring. Tends to produce tighter intervals when the noise is heteroscedastic.

In [ ]:
cqr_pipeline = UncertaintyExplanationPipeline(
    model=RandomForestRegressor(n_estimators=200, random_state=RNG),
    conformal_predictor=CQRConformalPredictor(
        lower_model=GradientBoostingRegressor(
            loss="quantile", alpha=0.05, random_state=RNG
        ),
        upper_model=GradientBoostingRegressor(
            loss="quantile", alpha=0.95, random_state=RNG
        ),
    ),
    confidence=0.9,
    random_state=RNG,
)
cqr_pipeline.fit(X_train, y_train)

lo_cqr, up_cqr = cqr_pipeline.predict(X_test)
cov_cqr = ((lo_cqr <= y_test) & (y_test <= up_cqr)).mean()
print(f"CQR coverage:    {cov_cqr:.2%}   (target ≥ 90 %)")
print(f"CQR mean width:  {(up_cqr - lo_cqr).mean():.2f}")

_ = cqr_pipeline.explain(X_test.iloc[:50], plot_kind="bar")

## Recap

- The pipeline gives valid (>= 90 %) coverage with default settings on synthetic data.
- SHAP/PDP/LIME each explain a single scalar metric (`interval_width` here). Pick by need: SHAP is per-sample + global, PDP is global only, LIME is per-sample only.
- Swap conformal backends by passing `conformal_predictor=...` — no other code changes needed.